# Importando bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import time
from sklearn.metrics import mean_absolute_error

## Carregando e explorando os dados

In [ ]:
df_features = pd.read_csv('./dataset/features/features.csv')
df_treino = pd.read_csv('./dataset/train/train.csv')
df_lojas = pd.read_csv('./dataset/stores.csv')

df_treino = pd.merge(df_treino, df_lojas, on='Store', how='left')
df_treino = pd.merge(df_treino, df_features, on=['Store', 'Date', 'IsHoliday'], how='left')

df_teste = pd.read_csv('./dataset/test/test.csv')
df_teste = pd.merge(df_teste, df_lojas, on='Store', how='left')
df_teste = pd.merge(df_teste, df_features, on=['Store', 'Date', 'IsHoliday'], how='left')

display(df_treino.head().style.hide(axis="index"))

In [ ]:
df_treino.describe(include='all')

In [ ]:
print(df_treino.isna().sum())

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(15, 6))
df_temporal = df_treino.groupby('Date')['Weekly_Sales'].sum().reset_index()
df_temporal['Date'] = pd.to_datetime(df_temporal['Date'])
sns.lineplot(data=df_temporal, x='Date', y='Weekly_Sales', color='blue')
plt.title('Evolução das vendas semanais totais')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_treino, x='IsHoliday', y='Weekly_Sales', showfliers=False)
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_treino, x='Size', y='Weekly_Sales', hue='Type', alpha=0.5)
plt.title('Relação entre tamanho da loja e vendas')
plt.show()

In [ ]:
df_treino['Date'] = pd.to_datetime(df_treino['Date'])
df_dezembro = df_treino[df_treino['Date'].dt.month == 12]
top_depts_natal = df_dezembro.groupby('Dept')['Weekly_Sales'].median().nlargest(10).reset_index()
print("Top 10 Departamentos no Natal:")
print(top_depts_natal)

vendas_negativas = df_treino[df_treino['Weekly_Sales'] < 0]
print(f"Total de registros negativos: {len(vendas_negativas)}")

## Preparando treinamento

In [ ]:
df_treino['Date'] = pd.to_datetime(df_treino['Date'])
df_treino['Year'] = df_treino['Date'].dt.year
df_treino['Month'] = df_treino['Date'].dt.month
df_treino['Week'] = df_treino['Date'].dt.isocalendar().week.astype(int)
df_treino['IsHoliday'] = df_treino['IsHoliday'].astype(int)
df_treino = df_treino.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)
df_treino['Sales_Previous_Year_52'] = df_treino.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
df_treino['Sales_Previous_Year_53'] = df_treino.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(53)
df_treino['Sales_Previous_Year_51'] = df_treino.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51)
df_treino = df_treino.dropna(subset=['Sales_Previous_Year_52', 'Sales_Previous_Year_51', 'Sales_Previous_Year_53'])
df_treino = pd.get_dummies(df_treino, columns=['Type'], drop_first=False)
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df_treino[markdown_cols] = df_treino[markdown_cols].fillna(0)
df_treino['Is_Refund'] = (df_treino['Weekly_Sales'] < 0).astype(int)

features = ['Store', 'Dept', 'IsHoliday', 'Size', 'Year', 'Month', 'Week', 'Sales_Previous_Year_53', 'Sales_Previous_Year_52', 'Sales_Previous_Year_51', 'Is_Refund', 'Type_A', 'Type_B', 'Type_C', 'CPI']
alvo = 'Weekly_Sales'

data_corte = '2012-06-01'
df_treino_modelo = df_treino[df_treino['Date'] < data_corte]
df_validacao_modelo = df_treino[df_treino['Date'] >= data_corte]

X_train = df_treino_modelo[features]
y_train = df_treino_modelo[alvo]
X_val = df_validacao_modelo[features]
y_val = df_validacao_modelo[alvo]

print(f"Linhas para treino: {df_treino_modelo.shape[0]}")
print(f"Linhas para validação: {df_validacao_modelo.shape[0]}")

## Treinamento do modelo

In [ ]:
print("Iniciando o treinamento...")
tempo_inicio = time.time()
modelo = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)
tempo_fim = time.time()
print(f"Treinamento concluído em {(tempo_fim - tempo_inicio)/60:.2f} minutos!")

## Avaliação do modelo

In [ ]:
previsoes = modelo.predict(X_val)
erro_mae = mean_absolute_error(y_val, previsoes)
print(f"O erro médio do modelo (MAE) é de: ${erro_mae:,.2f}")

# Aplicando modelo no conjunto de teste

In [ ]:
df_teste['Date'] = pd.to_datetime(df_teste['Date'])
df_treino_temp = df_treino[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Size', 'CPI']].copy()
df_teste['Weekly_Sales'] = 0
df_treino_teste = pd.concat([df_treino_temp, df_teste], ignore_index=True)
df_treino_teste = df_treino_teste.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)
df_treino_teste['Sales_Previous_Year_52'] = df_treino_teste.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)
df_treino_teste['Sales_Previous_Year_53'] = df_treino_teste.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(53)
df_treino_teste['Sales_Previous_Year_51'] = df_treino_teste.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(51)

data_inicio_teste = df_teste['Date'].min()
df_teste_processado = df_treino_teste[df_treino_teste['Date'] >= data_inicio_teste].copy()
df_teste_processado['Year'] = df_teste_processado['Date'].dt.year
df_teste_processado['Month'] = df_teste_processado['Date'].dt.month
df_teste_processado['Week'] = df_teste_processado['Date'].dt.isocalendar().week.astype(int)
df_teste_processado['IsHoliday'] = df_teste_processado['IsHoliday'].astype(int)
df_teste_processado = pd.get_dummies(df_teste_processado, columns=['Type'], drop_first=False)
df_teste_processado[markdown_cols] = df_teste_processado[markdown_cols].fillna(0)
df_teste_processado['Is_Refund'] = 0

X_test = df_teste_processado[features]
X_test = X_test.fillna(0)
previsoes = modelo.predict(X_test)

df_submissao = pd.DataFrame()
df_submissao['Id'] = df_teste_processado['Store'].astype(str) + '_' + df_teste_processado['Dept'].astype(str) + '_' + df_teste_processado['Date'].dt.strftime('%Y-%m-%d')
df_submissao['Weekly_Sales'] = previsoes
df_submissao.to_csv('submission.csv', index=False)
print("Submissão gerada com sucesso")